# Converting schema to clean human-readable version

## Read the schema (of unmapped entity)

In [1]:
import yaml
from typing import Any, Dict

In [10]:
## load the schema from the YAML file

path_schema = r'schema_unmapped.yaml'  
with open(path_schema, 'r') as f:
    schema = yaml.safe_load(f)
    
schema  # Now a Python dict

{'$schema': 'http://json-schema.org/draft-07/schema#',
 'title': 'UnmappedEntity',
 'type': 'object',
 'description': 'Staging schema for raw, unstructured data ingestion. All fields capture free-text inputs before data normalization.',
 'required': ['technology_name'],
 'properties': {'technology_name': {'type': 'string',
   'description': 'The literal unparsed common name or label of the technology as specified in the raw source material.'},
  'technology': {'type': 'object',
   'description': 'Contains raw descriptive properties detailing the engineering, structural, and hardware context of the unmapped entity.',
   'properties': {'technology_description': {'type': 'string',
     'description': 'Free-text engineering parameters describing the technical baseline of the asset.'},
    'technology_type': {'type': 'string',
     'description': 'The structural archetype classifying how the physical asset interacts with energy streams.',
     'examples': ['conversion', 'storage', 'distribu

In [ ]:
import yaml


# Force all strings to use standard quoted/plain inline style
def plain_str_presenter(dumper, data):
    # If it has special characters, wrap it in double quotes, otherwise leave it plain
    if any(char in data for char in [":", "{", "}", "[", "]", ",", "&", "*", "#", "?", "|", "-", "<", ">", "=", "!", "%", "@", "`"]):
        return dumper.represent_scalar("tag:yaml.org,2002:str", data, style='"')
    return dumper.represent_scalar("tag:yaml.org,2002:str", data)


yaml.add_representer(str, plain_str_presenter)

def json_schema_to_dict(schema):
    """Recursively converts a JSON schema into a clean, human-readable dictionary layout."""
    output = {}
    properties = schema.get("properties", {})
    required_fields = schema.get("required", [])

    for field_name, field_info in properties.items():
        field_type = field_info.get("type", "any")
        description = field_info.get("description", "No description provided.")

        # 1. Determine the status tag for the key parentheses
        tag = " (Required)" if field_name in required_fields else ""
        key_name = f"{field_name}{tag}"

        # 2. Determine the structural type prefix for the description text
        if field_type == "object":
            type_prefix = "[Object]"
        elif field_type == "array":
            item_type = field_info.get("items", {}).get("type", "any")
            type_prefix = f"[Array of {item_type.capitalize()}s]"
        else:
            type_prefix = f"[{field_type.capitalize()}]"

        # Combine type prefix with the base description string
        full_description = f"{type_prefix} {description}"

        # 3. Handle child objects or child arrays of objects recursively
        if field_type == "object" and "properties" in field_info:
            output[key_name] = {
                "description": full_description,
                "properties": json_schema_to_dict(field_info),
            }
        elif field_type == "array" and "properties" in field_info.get(
            "items", {}
        ):
            output[key_name] = {
                "description": full_description,
                "item_properties": json_schema_to_dict(field_info["items"]),
            }
        else:
            # Append examples as inline hints if they exist
            if "examples" in field_info:
                output[
                    key_name
                ] = f"{full_description} Suggested: {field_info['examples']}"
            else:
                output[key_name] = full_description

    return output


def convert_file_to_yaml(input_yaml_path, output_yaml_path):
    """Reads a JSON Schema file and writes a clean, 4-space indented human YAML file."""
    with open(input_yaml_path, "r", encoding="utf-8") as f:
        schema_data = yaml.safe_load(f)

    title = schema_data.get("title", "Schema Documentation")
    global_desc = schema_data.get("description", "")

    human_structure = {
        title: {
            "description": global_desc,
            "fields": json_schema_to_dict(schema_data),
        }
    }

    with open(output_yaml_path, "w", encoding="utf-8") as f:
        yaml.dump(
            human_structure,
            f,
            sort_keys=False,
            allow_unicode=True,
            default_flow_style=False,
            width=float("inf"),  # Disables auto-line-wrapping entirely
            indent=4,  # Forces 4 spaces indentation globally
        )

    print(f"Successfully generated clean human YAML at: {output_yaml_path}")

In [41]:

# if __name__ == "__main__":
input_file = "schema_unmapped.yaml"
output_file = "human_documentation.yaml"
print(
    f"Point the script to your '{input_file}' to generate '{output_file}'."
)
convert_file_to_yaml(input_file, output_file)

Point the script to your 'schema_unmapped.yaml' to generate 'human_documentation.yaml'.
Successfully generated clean human YAML at: human_documentation.yaml
